## Overview

This notebook will show you how to create and query a table or DataFrame that you uploaded to DBFS. [DBFS](https://docs.databricks.com/user-guide/dbfs-databricks-file-system.html) is a Databricks File System that allows you to store data for querying inside of Databricks. This notebook assumes that you have a file already inside of DBFS that you would like to read from.

This notebook is written in **Python** so the default cell type is Python. However, you can use different languages by using the `%LANGUAGE` syntax. Python, Scala, SQL, and R are all supported.

### This is to see list of files in the location , 
- %fs is file system-unix
- %md is for comment
- %python is for python code
- %r is for R code
- %sql is for sql code

In [0]:

%fs ls /FileStore/tables


path,name,size
dbfs:/FileStore/tables/restaurant-1.json,restaurant-1.json,914930
dbfs:/FileStore/tables/restaurant-2.json,restaurant-2.json,914930
dbfs:/FileStore/tables/restaurant-3.json,restaurant-3.json,315
dbfs:/FileStore/tables/restaurant-4.json,restaurant-4.json,914930
dbfs:/FileStore/tables/restaurant.json,restaurant.json,914930
dbfs:/FileStore/tables/restaurant_Simple_data-1.json,restaurant_Simple_data-1.json,213
dbfs:/FileStore/tables/restaurant_Simple_data.json,restaurant_Simple_data.json,215
dbfs:/FileStore/tables/restaurant_data-1.json,restaurant_data-1.json,315
dbfs:/FileStore/tables/restaurant_data.json,restaurant_data.json,315
dbfs:/FileStore/tables/restaurant_data1-1.json,restaurant_data1-1.json,458


### Read the JSON FILE

In [0]:

# File location and type
file_location = "/FileStore/tables/restaurant_data1-3.json"
file_type = "json"

# CSV options
infer_schema = "false"
first_row_is_header = "false"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
# If we have multiple file, we can give multiple file inside  "spark.read.format(multiple file)""
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

#Here df wont be saved as Python dataframe/table structure

# Another way to read the json file
#df = spark.read.json('/FileStore/tables/restaurant.json')


In [0]:
#This is to see the data in json file loaded df data, It stores data based on header ascending order
display(df)

business_id,comment,review_id,stars,user_id
BI1,ok good,1,1,UI1
BI1,ok average,1,1,UI1
BI1,Exccellent,1,1,UI1
BI1,bad,1,1,UI1
BI1,poor,1,1,UI1


In [0]:
#This is to read json file as text file and we can see it in text form via display(df_txt), each row will show complete 1 col vlaue 
df_txt = spark.read.text('/FileStore/tables/restaurant_data1-3.json')
display(df_txt)

value
"{""review_id"":""1"", ""user_id"":""UI1"" ,""business_id"":""BI1"", ""stars"":1, ""comment"":""ok good"" } ,"
"{""review_id"":""2"", ""user_id"":""UI2"" ,""business_id"":""BI2"", ""stars"":2, ""comment"":""ok average"" } ,"
"{""review_id"":""3"", ""user_id"":""UI3"" ,""business_id"":""BI3"", ""stars"":3, ""comment"":""Exccellent"" } ,"
"{""review_id"":""4"", ""user_id"":""UI4"" ,""business_id"":""BI4"", ""stars"":4, ""comment"":""bad"" } ,"
"{""review_id"":""5"", ""user_id"":""UI5"" ,""business_id"":""BI5"", ""stars"":5, ""comment"":""poor"" }"


In [0]:
#This is to see the schema of the json file
df.printSchema()

root
 |-- business_id: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: long (nullable = true)
 |-- user_id: string (nullable = true)



In [0]:
df.show()

+-----------+----------+---------+-----+-------+
|business_id|   comment|review_id|stars|user_id|
+-----------+----------+---------+-----+-------+
|        BI1|   ok good|        1|    1|    UI1|
|        BI2|ok average|        2|    2|    UI2|
|        BI3|Exccellent|        3|    3|    UI3|
|        BI4|       bad|        4|    4|    UI4|
|        BI5|      poor|        5|    5|    UI5|
+-----------+----------+---------+-----+-------+



In [0]:
#Select column
df.select("business_id").show()

+-----------+
|business_id|
+-----------+
|        BI1|
|        BI2|
|        BI3|
|        BI4|
|        BI5|
+-----------+



### Create a temporary view on Json file

In [0]:
# Create a view or table, this is python code only

temp_table_name = "restaurant_json"

df.createOrReplaceTempView(temp_table_name)

In [0]:
%sql
 
/* Query the created temp table in a SQL cell */

select * from `restaurant_json`

business_id,comment,review_id,stars,user_id
BI1,ok good,1,1,UI1
BI2,ok average,2,2,UI2
BI3,Exccellent,3,3,UI3
BI4,bad,4,4,UI4
BI5,poor,5,5,UI5


### Create PERM table on JSON file df

In [0]:
%sql
 
/* Delete the perm table in a SQL cell if its already created earlier */
drop table restaurant_json_pt;


In [0]:
# With this registered as a temp view, it will only be available to this particular notebook. If you'd like other users to be able to query this table, you can also create a table from the DataFrame.
# Once saved, this table will persist across cluster restarts as well as allow various users across different notebooks to query this data.
# To do so, choose your table name and uncomment the bottom line.


permanent_table_name = "restaurant_json_pt"

df.write.format("parquet").saveAsTable(permanent_table_name)

In [0]:
%sql

 
/* Query the perm table in a SQL cell */

select * from restaurant_json_pt; 


business_id,comment,review_id,stars,user_id
BI1,ok good,1,1,UI1
BI2,ok average,2,2,UI2
BI3,Exccellent,3,3,UI3
BI4,bad,4,4,UI4
BI5,poor,5,5,UI5


### Create JSON file from file system(fs) in \tmp folder inside dbfs with name as test.json

In [0]:
dbutils.fs.put("/tmp/test.json", """
{"string":"string1", "int":1, "array":[1,2,3], "dict":{"key":"values1"}}
{"string":"string2", "int":2, "array":[2,3,4], "dict":{"key":"values2"}}
{"string":"string3", "int":3, "array":[3,4,5], "dict":{"key":"values3", "extra_key":"extra_values3"}}
""", True)

Wrote 249 bytes.
Out[70]: True

In [0]:
%fs
ls /tmp/

path,name,size
dbfs:/tmp/hive/,hive/,0
dbfs:/tmp/test.json,test.json,249


In [0]:
#This is to read json file 
df_test = spark.read.json('/tmp/test.json')
display(df_test)

array,dict,int,string
"List(1, 2, 3)","List(null, values1)",1,string1
"List(2, 3, 4)","List(null, values2)",2,string2
"List(3, 4, 5)","List(extra_values3, values3)",3,string3


In [0]:
#To see schema
df_test.printSchema()

root
 |-- array: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- dict: struct (nullable = true)
 |    |-- extra_key: string (nullable = true)
 |    |-- key: string (nullable = true)
 |-- int: long (nullable = true)
 |-- string: string (nullable = true)



In [0]:
# Create a view or table, this is python code only

temp_table_name = "temp_test_json"

df_test.createOrReplaceTempView(temp_table_name)

In [0]:
%sql
 
/* Query the created temp table in a SQL cell */


select * from temp_test_json;

array,dict,int,string
"List(1, 2, 3)","List(null, values1)",1,string1
"List(2, 3, 4)","List(null, values2)",2,string2
"List(3, 4, 5)","List(extra_values3, values3)",3,string3


### Multiline NESTED JSON file data

In [0]:
# Here '[' at the beginning of values and ',' added  after each line
dbutils.fs.put("/tmp/multiline.json", """
[
  {"string":"string1", "int":1, "array":[1,2,3], "dict":{"key":"values1"}},
  {"string":"string2", "int":2, "array":[2,3,4], "dict":{"key":"values2"}},

    { 
      "string":"string3", 
      "int":3, 
      "array":[3,
               4,
              5], 
      "dict":{
              "key":"values3", 
              "extra_key":"extra_values3"
              }
   }
  ] """, True)

Wrote 375 bytes.
Out[85]: True

In [0]:
#This is to read  multiline json file without multiline option, this gives curropted file
df_multiline1 = spark.read.json("/tmp/multiline.json")
display(df_multiline1)

_corrupt_record,array,dict,int,string
null,"List(1, 2, 3)",List(values1),1,string1
null,"List(2, 3, 4)",List(values2),2,string2
{,null,null,null,null
"""string"":""string3"",",null,null,null,null
"""int"":3,",null,null,null,null
"""array"":[3,",null,null,null,null
"4,",null,null,null,null
"5],",null,null,null,null
"""dict"":{",null,null,null,null
"""key"":""values3"",",null,null,null,null


In [0]:
#This is to read  multiline json file with multiline option
df_multiline = spark.read.option('multiline',"true").json("/tmp/multiline.json")


In [0]:
display(df_multiline)

array,dict,int,string
"List(1, 2, 3)","List(null, values1)",1,string1
"List(2, 3, 4)","List(null, values2)",2,string2
"List(3, 4, 5)","List(extra_values3, values3)",3,string3


### Create RDD for customised (NESTED) JSON data
#### Example 1

In [0]:
# an RDD{String} storing one JSON object per string
my_data = ['{"name":"Prabha", "address":{"city": "bangalore", "state": "karnataka"}}']
#Create RDD by "sc.parallelize"
my_rdd =  sc.parallelize(my_data)

my_df =  spark.read.json(my_rdd)
my_df.printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- name: string (nullable = true)



In [0]:
#Nested field value reading by using .select in pyspark
display(my_df.select("address.city","address.state","name"))

city,state,name
bangalore,karnataka,Prabha


#### Example 2

In [0]:
# an RDD{String} storing one JSON object per string
#data = ['{"name":"Prabha", "address":{"city": "bangalore", "state": "karnataka"}}']

#Create RDD by "sc.parallelize"
my_rdd2 =  sc.parallelize(
  ( 
"""
{ "id": "123",
"name": "raj",
"age":37,
"eyecolor": "brown"
} """,

"""
{ "id": "234",
"name": "prabha",
"age":50,
"eyecolor": "black"
} """,

"""
{ "id": "345",
"name": "atharv",
"age":2,
"eyecolor": "red"
} """
   )
)


In [0]:
my_df2 =  spark.read.json(my_rdd2)
my_df2.printSchema()

root
 |-- age: long (nullable = true)
 |-- eyecolor: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)



In [0]:
display(my_df2)

age,eyecolor,id,name
37,brown,123,raj
50,black,234,prabha
2,red,345,atharv


### Create JSON file from Dataframe

- Here it creates folder and inside that it will write mutliple file

- Writing json file using dataframe with "write"
- Using mode('append') --> This append new files everytime
- Using mode('overwrite')  --> This replaces all file with 1 new files everytime

In [0]:
my_df2.write.mode('append').json('/FileStore/tables/write_sample_json')
#my_df2.write.mode('overwrite').json('/FileStore/tables/write_sample_json')

In [0]:
%fs
ls /FileStore/tables/write_sample_json

path,name,size
dbfs:/FileStore/tables/write_sample_json/_SUCCESS,_SUCCESS,0
dbfs:/FileStore/tables/write_sample_json/_committed_3235562913639353211,_committed_3235562913639353211,739
dbfs:/FileStore/tables/write_sample_json/_committed_3463396175849432305,_committed_3463396175849432305,1463
dbfs:/FileStore/tables/write_sample_json/_committed_4203614049892201446,_committed_4203614049892201446,384
dbfs:/FileStore/tables/write_sample_json/_committed_4482885185815528563,_committed_4482885185815528563,384
dbfs:/FileStore/tables/write_sample_json/_committed_792461841267768408,_committed_792461841267768408,380
dbfs:/FileStore/tables/write_sample_json/_committed_896021321534176152,_committed_896021321534176152,746
dbfs:/FileStore/tables/write_sample_json/_started_3235562913639353211,_started_3235562913639353211,0
dbfs:/FileStore/tables/write_sample_json/_started_3463396175849432305,_started_3463396175849432305,0
dbfs:/FileStore/tables/write_sample_json/_started_4203614049892201446,_started_4203614049892201446,0
